# Loan data audit

Audit the raw extract with `ml-gearbox` before choosing transformations. `loan_status` has binary codes; their business meaning is undocumented. Report class 1 without assuming it means default.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from ml_gearbox import audit as ga

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data/raw/data.csv"
assert RAW_PATH.is_file(), f"Missing raw data: {RAW_PATH}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.audit import run_loan_audit

In [2]:
df = pd.read_csv(RAW_PATH)
display(ga.schema_report(df))
display(ga.missingness_report(df))

,dtype,missing_count,missing_pct,n_unique,unique_pct
emp_length,str,35934,6.927624,12,0.002313
revol_util,float64,413,0.079621,1181,0.227682
dti,float64,372,0.071717,7051,1.359344
inq_last_6mths,float64,1,0.000193,7,0.001350
installment,float64,0,0.000000,52757,10.170887
int_rate,float64,0,0.000000,241,0.046462
loan_amnt,float64,0,0.000000,1548,0.298435
term,str,0,0.000000,2,0.000386
loan_status,int64,0,0.000000,2,0.000386
annual_inc,float64,0,0.000000,31497,6.072226


,feature,missing_count,percentage
0,emp_length,35934,6.927624
1,revol_util,413,0.079621
2,dti,372,0.071717
3,inq_last_6mths,1,0.000193


In [3]:
report = run_loan_audit(df)
display(report["target"])
display(pd.Series(report["duplicates"], name="count").to_frame())
display(report["numeric_profile"])

,class,count,percentage
0,0,402449,77.587111
1,1,116257,22.412889


,count
full_row_duplicates,0


,count,mean,std,min,25%,50%,75%,max,median,skew,n_unique
annual_inc,518706.0,78975.296566,77463.678725,0.00,48000.00,66000.00,95000.0000,10999200.00,66000.00,45.966326,31497
dti,518334.0,18.778012,14.522484,-1.00,11.82,17.83,24.5800,999.00,17.83,30.983804,7050
pub_rec,518706.0,0.244522,0.629787,0.00,0.00,0.00,0.0000,61.00,0.00,8.887501,30
delinq_2yrs,518706.0,0.327895,0.908625,0.00,0.00,0.00,0.0000,29.00,0.00,5.695472,26
pub_rec_bankruptcies,518706.0,0.154616,0.400768,0.00,0.00,0.00,0.0000,9.00,0.00,3.090781,10
inq_last_6mths,518705.0,0.587785,0.870538,0.00,0.00,0.00,1.0000,5.00,0.00,1.725519,6
mort_acc,518706.0,1.551945,1.849522,0.00,0.00,1.00,3.0000,51.00,1.00,1.618283,32
loan_status,518706.0,0.224129,0.417008,0.00,0.00,0.00,0.0000,1.00,0.00,1.323103,2
fico_range_low,518706.0,698.045174,33.586796,660.00,670.00,690.00,715.0000,845.00,690.00,1.235967,38
total_acc,518706.0,24.599029,12.231060,2.00,16.00,23.00,31.0000,176.00,23.00,1.060101,131


In [4]:
display(report["range_validation"])
display(report["count_integrality"])
display(report["cross_field_validation"])
display(pd.Series({"negative_dti_count": report["negative_dti_count"]}, name="count").to_frame())

,feature,check,violation_count,violation_pct
0,loan_amnt,min >= 0,0,0.0
1,int_rate,min >= 0,0,0.0
2,installment,min >= 0,0,0.0
3,annual_inc,min >= 0,0,0.0
4,dti,min >= 0,2,0.0
5,delinq_2yrs,min >= 0,0,0.0
6,fico_range_low,min >= 300,0,0.0
7,fico_range_low,max <= 850,0,0.0
8,inq_last_6mths,min >= 0,0,0.0
9,pub_rec,min >= 0,0,0.0


,feature,non_integer_count
0,delinq_2yrs,0
1,inq_last_6mths,0
2,pub_rec,0
3,total_acc,0
4,mort_acc,0
5,pub_rec_bankruptcies,0


,rule,violation_count,violation_pct,note
0,pub_rec_bankruptcies <= pub_rec,0,0.0,None


,count
negative_dti_count,2


In [5]:
display(report["hidden_missingness"])
display(report["category_normalization"])
display(report["infinite_values"])
display(report["constant_features"])

,feature,hidden_missing_count,percentage


,feature,original_unique,normalized_unique


,feature,infinite_count


,feature,dominant_percentage,is_strictly_constant
0,pub_rec_bankruptcies,85.653723,False
1,pub_rec,80.742656,False
2,delinq_2yrs,80.381179,False
3,loan_status,77.587111,False
4,term,77.531203,False
5,inq_last_6mths,59.876886,False
6,mort_acc,40.758156,False
7,emp_length,33.483708,False
8,fico_range_low,8.979268,False
9,loan_amnt,7.863607,False


Keep raw data unchanged. Review flagged records before preprocessing. Negative DTI becomes missing in the core preparation strategy; high valid values remain available. Class meaning and feature availability at the intended prediction time require source documentation.